# A first look at CIFAR-100

**Goal:** load the images, understand the labels, and record a few observations before we build a model.

Work through the notebook from top to bottom, or choose **Kernel → Restart Kernel and Run All Cells**. The first run downloads about **169 MB**; later runs reuse the cached file. No GPU is needed.

We explore only the **training set** here. Leave the official test set untouched for later evaluation. Corrupted images, SVHN, calibration, and modeling will come later.

Dataset: [CIFAR-100, University of Toronto](https://www.cs.toronto.edu/~kriz/cifar.html), by Alex Krizhevsky, Vinod Nair, and Geoffrey Hinton. Images have a **fine label** (one of 100 classes) and a **coarse label** (one of 20 broader groups).


## 1. Import our tools

NumPy handles image arrays, pandas makes tables, and Matplotlib draws the plots.
The remaining imports are part of Python and handle the download and archive.


In [ ]:
from pathlib import Path
import hashlib
import pickle
import tarfile
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


## 2. Download once, then load the training data

The archive is cached in `.cache/sdm-vision-reliability` under your home directory, outside the Git repository. Change `DATA_DIR` if you already keep data somewhere else.

We check the archive against the [checksum used by torchvision](https://github.com/pytorch/vision/blob/v0.25.0/torchvision/datasets/cifar.py) before reading its pickle files. Only use this verified archive. If a download is interrupted, the next run downloads it again. A slow first download is normal; you can also put the official `cifar-100-python.tar.gz` file in `DATA_DIR` yourself.


In [ ]:
DATA_DIR = Path.home() / ".cache" / "sdm-vision-reliability"
DATA_DIR.mkdir(parents=True, exist_ok=True)
archive_path = DATA_DIR / "cifar-100-python.tar.gz"
url = "https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz"
expected_md5 = "eb9058c3a382ffc7106e4002c42a8d85"

if not archive_path.exists():
    print("Downloading CIFAR-100 (about 169 MB)...")
    temporary_path = archive_path.with_suffix(".download")
    urlretrieve(url, temporary_path)
    temporary_path.replace(archive_path)

with archive_path.open("rb") as file:
    checksum = hashlib.file_digest(file, "md5").hexdigest()
if checksum != expected_md5:
    raise ValueError("The archive checksum does not match. Remove the cached archive and rerun this cell.")
print("Archive verified. Ready to load training images.")


In [ ]:
with tarfile.open(archive_path, "r:gz") as archive:
    training = pickle.load(archive.extractfile("cifar-100-python/train"), encoding="latin1")
    metadata = pickle.load(archive.extractfile("cifar-100-python/meta"), encoding="latin1")

# The archive stores each image as three flattened color channels.
# Rearrange it to (image, height, width, color channel) for plotting.
images = training["data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
fine_labels = np.array(training["fine_labels"])
coarse_labels = np.array(training["coarse_labels"])
class_names = metadata["fine_label_names"]
group_names = metadata["coarse_label_names"]

labels = pd.DataFrame({
    "class": np.array(class_names)[fine_labels],
    "group": np.array(group_names)[coarse_labels],
})
labels.head()


## 3. What did we load?

Each image is **32 × 32 pixels**, with red, green, and blue channels. Pixel values are integers from 0 (dark) to 255 (bright). We keep them in their original form for this first look.


In [ ]:
pd.Series({
    "training images": len(images),
    "image shape (height, width, channels)": images.shape[1:],
    "pixel type": str(images.dtype),
    "smallest pixel value": int(images.min()),
    "largest pixel value": int(images.max()),
    "classes": labels["class"].nunique(),
    "broader groups": labels["group"].nunique(),
}, name="CIFAR-100 training set").to_frame()


## 4. See some images and their names

The fixed seed makes this gallery repeatable. Try changing `SEED` and rerunning this cell to see another sample. These tiny images are enlarged without smoothing so we can see their actual pixels.


In [ ]:
sample_indices = np.random.default_rng(SEED).choice(len(images), size=25, replace=False)
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for ax, index in zip(axes.flat, sample_indices):
    ax.imshow(images[index], interpolation="nearest")
    ax.set_title(labels.loc[index, "class"].replace("_", " "), fontsize=10)
    ax.axis("off")
fig.suptitle("25 training images with their class names", fontsize=15)
plt.tight_layout()
plt.show()


**Discuss:** Which images are easy to recognize? Which are ambiguous at this resolution? Does the label tell the whole story?


## 5. Explore one class

Run the next cell to see the available names. Then replace `CLASS_NAME` in the gallery cell with a name that interests you.


In [ ]:
print(", ".join(class_names))


In [ ]:
CLASS_NAME = "apple"

if CLASS_NAME not in class_names:
    raise ValueError("Choose a class name from the list above.")
class_indices = labels.index[labels["class"] == CLASS_NAME].to_numpy()
chosen = np.random.default_rng(SEED).choice(class_indices, size=12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(8, 6))
for ax, index in zip(axes.flat, chosen):
    ax.imshow(images[index], interpolation="nearest")
    ax.axis("off")
fig.suptitle(f"{CLASS_NAME.replace('_', ' ')} — 12 examples from the same class", fontsize=14)
plt.tight_layout()
plt.show()


**Discuss:** Within this class, what changes—background, color, viewpoint, lighting, or object size? What stays similar?


## 6. Are the labels balanced?

Count images in each fine class and each broader group. A balanced dataset has the same number of examples per label; that does not mean every class is equally easy to recognize.


In [ ]:
class_counts = labels.groupby(["group", "class"]).size().rename("images").reset_index()
display(class_counts.head(10))
print(f"Images per class: {class_counts['images'].min()} to {class_counts['images'].max()}")
print(f"All {len(class_counts)} classes have equal counts: {class_counts['images'].nunique() == 1}")


In [ ]:
group_counts = labels["group"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 7))
group_counts.plot.barh(ax=ax, color="#287c8e")
ax.set_title("Training images per broader group")
ax.set_xlabel("Number of images")
ax.set_ylabel("")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 7. A simple look at color values

To keep this quick, sample 2,000 training images. Each histogram combines all pixels in one color channel. These plots describe the sampled images; they do not measure model reliability or tell us whether a particular label is correct.


In [ ]:
pixel_indices = np.random.default_rng(SEED).choice(len(images), size=2000, replace=False)
pixel_sample = images[pixel_indices]

fig, ax = plt.subplots(figsize=(9, 4))
for channel, name, color in zip(range(3), ["Red", "Green", "Blue"], ["#c44e52", "#25845a", "#3574ad"]):
    ax.hist(pixel_sample[..., channel].ravel(), bins=32, range=(0, 256),
            density=True, histtype="step", linewidth=1.5, label=name, color=color)
ax.set(title="Pixel values in 2,000 sampled training images", xlabel="Pixel value (0–255)", ylabel="Density")
ax.legend()
plt.tight_layout()
plt.show()


## 8. Record our first observations

Edit this Markdown cell with your own observations:

- **Something I noticed in the image galleries:** ...
- **A class I explored and the variation I saw:** ...
- **What the class-count and color plots tell us:** ...
- **A question we should investigate next:** ...

Before modeling, agree as a team on the next experiment and evaluation plan. Keep the test set for final evaluation. Class balance and these plots alone do not establish accuracy, calibration, or reliability under distribution shift.
